# Noetia — Whisper Transcription Pipeline

Runs on a free GPU (T4) in Google Colab. Transcribes LibriVox audiobooks with word-level timestamps.

**Single book (Cells 1–7):**
Run Cells 1 → 2 → 3 (edit) → 4 → 5 → 6 → 7

**Batch mode — all books overnight (Cells 1–2–4–5–8):**
Run Cells 1 → 2 → 4 → 5 → 8. Skip Cells 3, 6, 7.

> Go to **Runtime → Change runtime type → T4 GPU → Save** before starting.
>
> If disconnected: re-run Cells 1–2–4–5, then re-run Cell 6 or Cell 8. Progress is saved to Drive.
>
> **Magnifica Humanitas** uses Vatican News audio (not LibriVox) and has pending rights — it cannot be processed with this pipeline.

**Check Drive before downloading (Cell 9):**
Run in a separate Colab session/tab from the active batch — just Cells 4 → 5 → 9 (no GPU, no Cell 8). Scans every merged VTT already in Drive for dropped chapters or Whisper hallucination loops, before you download/commit them.

In [ ]:
# CELL 1 — Check GPU
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print('GPU found!')
    for line in result.stdout.split('\n')[:12]:
        print(line)
else:
    print('NO GPU FOUND.')
    print('Go to: Runtime → Change runtime type → T4 GPU → Save, then re-run.')

In [ ]:
# CELL 2 — Install dependencies
!pip install -q openai-whisper
!apt-get install -q -y ffmpeg
print('Dependencies ready.')

In [ ]:
# CELL 3 — Configuration (single-book mode only — not used in Cell 8 batch)

LIBRIVOX_URL = 'https://librivox.org/don-quijote-vol-1-by-miguel-de-cervantes-saavedra/'
SLUG        = 'don-quijote-vol-1'
BOOK_DIR    = 'Don Quijote Vol. I'
LANGUAGE    = 'es'
MODEL       = 'medium'

print(f'Book  : {BOOK_DIR}')
print(f'Slug  : {SLUG}')
print(f'Model : {MODEL}  |  Language: {LANGUAGE}')

In [ ]:
# CELL 4 — Pipeline functions (no edits needed)
import json
import re
import subprocess
import urllib.parse
import urllib.request
from pathlib import Path

DRIVE_DIR   = Path('/content/drive/MyDrive/noetia-whisper')
GAP_SECONDS = 2.0


def fetch(url):
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0 Noetia-Whisper-Bot'})
    with urllib.request.urlopen(req, timeout=30) as r:
        return r.read().decode('utf-8', errors='replace')


def fetch_bytes(url, dest, label=''):
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0 Noetia-Whisper-Bot'})
    with urllib.request.urlopen(req, timeout=120) as r, open(dest, 'wb') as f:
        total = int(r.headers.get('Content-Length', 0))
        downloaded = 0
        while chunk := r.read(65536):
            f.write(chunk)
            downloaded += len(chunk)
            if total:
                pct = downloaded * 100 // total
                print(f'\r  {label}: {pct}%', end='', flush=True)
    print()


def scrape_librivox(librivox_url):
    print(f'Fetching: {librivox_url}')
    html = fetch(librivox_url)
    m = re.search(r'href="(https://archive\.org/compress/([^"]+)\.zip[^"]*)"', html, re.I)
    if m:
        identifier = re.search(r'archive\.org/(?:compress|download)/([^/?#]+)', m.group(1)).group(1)
        print(f'  Identifier: {identifier}')
        return identifier
    m = re.search(r'href="https://archive\.org/(?:compress|download)/([^/"]+)', html, re.I)
    if m:
        print(f'  Identifier: {m.group(1)}')
        return m.group(1)
    raise RuntimeError('Could not find archive.org identifier on LibriVox page')


def get_chapter_mp3s(identifier):
    meta = json.loads(fetch(f'https://archive.org/metadata/{identifier}'))
    files = meta.get('files', [])
    mp3s = [f for f in files if f.get('name', '').lower().endswith('.mp3') and
            f.get('format', '').lower() in ('mp3', 'vbr mp3', '')]
    kb64 = [f for f in mp3s if '64kb' in f.get('name', '').lower() or '64kb' in f.get('format', '').lower()]
    chosen = kb64 if kb64 else mp3s
    if not chosen:
        raise RuntimeError(f'No MP3 files found for: {identifier}')
    chosen.sort(key=lambda f: tuple(int(n) for n in re.findall(r'\d+', f.get('name', ''))))
    print(f'  Found {len(chosen)} chapters')
    return chosen, identifier


def download_chapters(identifier, files, audio_dir):
    audio_dir.mkdir(parents=True, exist_ok=True)
    base = f'https://archive.org/download/{identifier}'
    paths = []
    for i, f in enumerate(files):
        name = f['name']
        local = audio_dir / name
        if local.exists():
            print(f'  [{i+1}/{len(files)}] cached: {name}')
        else:
            print(f'  [{i+1}/{len(files)}] downloading: {name}')
            fetch_bytes(f'{base}/{urllib.parse.quote(name)}', local, label=name)
        paths.append(local)
    return paths


def transcribe_chapter(audio_path, vtt_dir, lang, model):
    vtt = vtt_dir / (audio_path.stem + '.vtt')
    if vtt.exists():
        print(f'  already done: {audio_path.name}')
        return vtt
    cmd = ['whisper', str(audio_path), '--model', model, '--language', lang,
           '--word_timestamps', 'True', '--output_format', 'vtt', '--output_dir', str(vtt_dir)]
    print(f'  transcribing: {audio_path.name} ...', end='', flush=True)
    result = subprocess.run(cmd, capture_output=True, text=True)
    combined = result.stdout + result.stderr
    if result.returncode != 0 or 'Skipping' in combined:
        print(f'\n  FAILED')
        print(combined[-1000:])
        raise RuntimeError(f'Whisper failed on {audio_path.name}')
    print(' done')
    if not vtt.exists():
        raise RuntimeError(f'VTT not written for {audio_path.name}')
    return vtt


def ts_to_sec(ts):
    parts = ts.strip().replace(',', '.').split(':')
    if len(parts) == 3:
        return int(parts[0]) * 3600 + int(parts[1]) * 60 + float(parts[2])
    return int(parts[0]) * 60 + float(parts[1])


def sec_to_ts(s):
    h, rem = divmod(s, 3600)
    m, sec = divmod(rem, 60)
    return f'{int(h):02d}:{int(m):02d}:{sec:06.3f}'


def shift_inline(text, offset):
    def rep(m):
        return f'<{sec_to_ts(ts_to_sec(m.group(1)) + offset)}>'
    return re.sub(r'<(\d{2}:\d{2}:\d{2}[\.,]\d{3})>', rep, text)


def parse_cues(vtt):
    cues = []
    for block in re.split(r'\n\n+', vtt.strip()):
        lines = block.strip().splitlines()
        for i, line in enumerate(lines):
            m = re.match(r'(\d{2}:\d{2}:\d{2}[\.,]\d{3})\s*-->\s*(\d{2}:\d{2}:\d{2}[\.,]\d{3})', line)
            if m:
                cues.append((ts_to_sec(m.group(1)), ts_to_sec(m.group(2)), '\n'.join(lines[i+1:])))
                break
    return cues


def merge_vtt_dir(vtt_dir, out_path):
    files = sorted(vtt_dir.glob('*.vtt'), key=lambda f: tuple(int(n) for n in re.findall(r'\d+', f.name)))
    if not files:
        raise RuntimeError(f'No VTT files in {vtt_dir}')
    all_cues, offset = [], 0.0
    for path in files:
        cues = parse_cues(path.read_text(encoding='utf-8'))
        if not cues:
            continue
        for s, e, p in cues:
            all_cues.append((s + offset, e + offset, shift_inline(p, offset)))
        offset += max(e for _, e, _ in cues) + GAP_SECONDS
        print(f'  + {path.name} ({len(cues)} cues)')
    lines = ['WEBVTT', '']
    for i, (s, e, p) in enumerate(all_cues, 1):
        lines += [str(i), f'{sec_to_ts(s)} --> {sec_to_ts(e)}', p, '']
    out_path.write_text('\n'.join(lines), encoding='utf-8')
    print(f'  Merged {len(all_cues)} cues → {out_path.name}')


def run_book(url, slug, book_dir, lang, model='medium'):
    merged = DRIVE_DIR / f'{slug}.merged.vtt'
    if merged.exists():
        print(f'  Already done: {book_dir}')
        return merged
    vtt_dir = DRIVE_DIR / book_dir
    vtt_dir.mkdir(parents=True, exist_ok=True)
    audio_dir = Path('/content/audio') / slug
    identifier = scrape_librivox(url)
    files, identifier = get_chapter_mp3s(identifier)
    audio_paths = download_chapters(identifier, files, audio_dir)
    print(f'  Transcribing {len(audio_paths)} chapters...')
    for i, audio in enumerate(audio_paths):
        print(f'  [{i+1}/{len(audio_paths)}]', end=' ')
        transcribe_chapter(audio, vtt_dir, lang, model)
    print('  Merging...')
    merge_vtt_dir(vtt_dir, merged)
    return merged


print('Functions loaded.')

In [ ]:
# CELL 5 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = Path('/content/drive/MyDrive/noetia-whisper')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
done = list(DRIVE_DIR.glob('*.merged.vtt'))
print(f'Drive mounted. Merged VTTs in Drive: {len(done)}')
for f in sorted(done):
    print(f'  {f.name}')

In [ ]:
# CELL 6 — Single-book pipeline (uses Cell 3 config)
merged = run_book(LIBRIVOX_URL, SLUG, BOOK_DIR, LANGUAGE, MODEL)
print(f'Done: {merged}')

In [ ]:
# CELL 7 — Download a merged VTT from Drive to your computer
from google.colab import files as colab_files

DOWNLOAD_SLUG = SLUG  # change to the slug you want, e.g. 'jane-eyre'

target = DRIVE_DIR / f'{DOWNLOAD_SLUG}.merged.vtt'
if target.exists():
    print(f'Downloading: {target.name}')
    colab_files.download(str(target))
else:
    available = [f.name for f in sorted(DRIVE_DIR.glob('*.merged.vtt'))]
    print(f'Not found: {target.name}')
    print(f'Available: {available}')

In [ ]:
# CELL 8 — Batch mode: all Spanish + English books
# Run AFTER Cells 1-2-4-5. Books already in Drive are skipped automatically.
# Safe to re-run after a disconnect — resumes where it left off.
#
# NOT included (cannot use LibriVox pipeline):
#   - Magnifica Humanitas: uses Vatican News audio + rights pending
#   - Genesis EN / Exodus EN: shared LibriVox recording with Leviticus — handle separately
#   - Ephesians EN / Philippians EN: shared recording with other epistles — handle separately

BATCH_BOOKS = [
    # (librivox_url, slug, book_dir, language)

    # ══════════════════════════════════════════════════════════════════════════
    # SPANISH BOOKS
    # ══════════════════════════════════════════════════════════════════════════

    # ── Long literary books ───────────────────────────────────────────────────
    ('https://librivox.org/don-quijote-vol-1-by-miguel-de-cervantes-saavedra/',           'don-quijote-vol-1',   'Don Quijote Vol. I',                 'es'),
    ('https://librivox.org/don-quijote-volume-2-by-miguel-de-cervantes-saavedra/',        'don-quijote-vol-2',   'Don Quijote Vol. II',                'es'),
    ('https://librivox.org/la-divina-comedia-by-dante-alighieri/',                        'la-divina-comedia',   'La Divina Comedia',                  'es'),
    ('https://librivox.org/crimen-y-castigo-by-fyodor-dostoyevsky/',                      'crimen-y-castigo',    'Crimen y Castigo',                   'es'),
    ('https://librivox.org/los-cuatro-jinetes-del-apocalipsis-by-vicente-blasco-ibanez/', 'cuatro-jinetes',      'Los Cuatro Jinetes del Apocalipsis', 'es'),

    # ── Medium literary books ─────────────────────────────────────────────────
    ('https://librivox.org/orgullo-y-prejuicio1-by-jane-austen/',                         'orgullo-y-prejuicio', 'Orgullo y Prejuicio',                'es'),
    ('https://librivox.org/pepita-jimenez-by-juan-valera/',                               'pepita-jimenez',      'Pepita Jiménez',                     'es'),
    ('https://librivox.org/platero-y-yo-by-juan-ramon-jimenez/',                          'platero-y-yo',        'Platero y yo',                       'es'),
    ('https://librivox.org/viaje-al-centro-de-la-tierra-by-jules-verne/',                 'viaje-al-centro',     'Viaje al Centro de la Tierra',       'es'),
    ('https://librivox.org/la-odisea-by-homero/',                                         'la-odisea',           'La Odisea',                          'es'),

    # ── Short literary books ──────────────────────────────────────────────────
    ('https://librivox.org/cuentos-de-la-selva-para-los-ninos-by-horacio-quiroga/',       'cuentos-de-la-selva', 'Cuentos de la Selva',                'es'),
    ('https://librivox.org/fabulas-y-verdades-by-rafael-pombo/',                          'fabulas-y-verdades',  'Fábulas y Verdades',                 'es'),
    ('https://librivox.org/el-gaucho-martin-fierro-by-jose-hernandez/',                   'martin-fierro',       'El Gaucho Martín Fierro',            'es'),

    # ── Spanish Bible (Reina-Valera) ──────────────────────────────────────────
    ('https://librivox.org/genesis-reina-valera-version/',                                'genesis-es',          'Génesis',                            'es'),
    ('https://librivox.org/bible-reina-valera-02-exodo-by-reina-valera/',                 'exodo-es',            'Éxodo',                              'es'),
    ('https://librivox.org/bible-reina-valera-nt-01-mateo-by-reina-valera/',              'mateo-es',            'Mateo',                              'es'),
    ('https://librivox.org/el-bible-reina-valera-nt-02-evangelio-segun-marcos-by-reina-valera/', 'marcos-es',    'Marcos',                             'es'),
    ('https://librivox.org/bible-reina-valera-nt-03-lucas-by-reina-valera/',              'lucas-es',            'Lucas',                              'es'),
    ('https://librivox.org/bible-reina-valera-nt-04-juan-by-reina-valera/',               'juan-es',             'Juan',                               'es'),
    ('https://librivox.org/bible-nt-06-romanos-by-reina-valera/',                         'romanos-es',          'Romanos',                            'es'),
    ('https://librivox.org/bible-reina-valera-nt-05-hechos-de-los-apostoles-by-reina-valera/', 'hechos-es',      'Hechos',                             'es'),
    ('https://librivox.org/bible-reina-valera-nt-27-apocalipsis-by-reina-valera/',        'apocalipsis-es',      'Apocalipsis',                        'es'),
    ('https://librivox.org/bible-reina-valera-1909-20-libro-de-los-proverbios-by-reina-valera/', 'proverbios-es', 'Proverbios',                        'es'),
    ('https://librivox.org/isaias-by-reina-valera/',                                      'isaias-es',           'Isaías',                             'es'),
    ('https://librivox.org/bible-reina-valera-antigua-19-salmos-by-reina-valera/',        'salmos-es',           'Salmos',                             'es'),
    ('https://librivox.org/carta-del-apostol-santiago-by-reina-valera/',                  'santiago-es',         'Santiago',                           'es'),
    ('https://librivox.org/1-corintios-reina-valera/',                                    '1-corintios-es',      '1 Corintios',                        'es'),
    ('https://librivox.org/hebreos-version-reina-valera/',                                'hebreos-es',          'Hebreos',                            'es'),
    ('https://librivox.org/bible-reina-valera-nt-10-la-epistola-del-apostol-san-pablo-a-los-efesios-by-reina-valera/', 'efesios-es', 'Efesios',        'es'),
    ('https://librivox.org/bible-reina-valera-nt-11-filipenses-by-reina-valera/',         'filipenses-es',       'Filipenses',                         'es'),

    # ══════════════════════════════════════════════════════════════════════════
    # ENGLISH CLASSICS
    # ══════════════════════════════════════════════════════════════════════════

    ('https://librivox.org/walden-by-henry-david-thoreau',                                'walden',              'Walden',                             'en'),
    ('https://librivox.org/the-meditations-of-marcus-aurelius/',                          'meditations',         'Meditations',                        'en'),
    ('https://librivox.org/jane-eyre-by-charlotte-bront/',                                'jane-eyre',           'Jane Eyre',                          'en'),
    ('https://librivox.org/pride-and-prejudice-by-jane-austen/',                          'pride-and-prejudice', 'Pride and Prejudice',                'en'),
    ('https://librivox.org/treasure-island-by-robert-louis-stevenson/',                   'treasure-island',     'Treasure Island',                    'en'),
    ('https://librivox.org/dracula-by-bram-stoker/',                                      'dracula',             'Dracula',                            'en'),
    ('https://librivox.org/anne-of-green-gables-by-lucy-maud-montgomery/',                'anne-of-green-gables','Anne of Green Gables',               'en'),
    ('https://librivox.org/frankenstein-or-modern-prometheus-by-mary-w-shelley/',         'frankenstein',        'Frankenstein',                       'en'),
    ('https://librivox.org/the-picture-of-dorian-gray-by-oscar-wilde/',                   'dorian-gray',         'The Picture of Dorian Gray',         'en'),
    ('https://librivox.org/the-scarlet-letter-by-nathaniel-hawthorne/',                   'scarlet-letter',      'The Scarlet Letter',                 'en'),
    ('https://librivox.org/the-strange-case-of-dr-jekyll-and-mr-hyde-by-robert-louis-stevenson/', 'jekyll-hyde', 'The Strange Case of Dr Jekyll and Mr Hyde', 'en'),
    ('https://librivox.org/tom-sawyer-by-mark-twain/',                                    'tom-sawyer',          'The Adventures of Tom Sawyer',       'en'),
    ('https://librivox.org/alices-adventures-in-wonderland-by-lewis-carroll/',            'alice-in-wonderland', "Alice's Adventures in Wonderland",   'en'),
    ('https://librivox.org/the-time-machine-by-hg-wells/',                                'time-machine',        'The Time Machine',                   'en'),

    # ══════════════════════════════════════════════════════════════════════════
    # ENGLISH BIBLE (KJV) — books with dedicated LibriVox recordings
    # ══════════════════════════════════════════════════════════════════════════

    ('https://librivox.org/the-book-of-psalms-king-james-version/',                       'psalms-en',           'Psalms',                             'en'),
    ('https://librivox.org/proverbs-king-james-version/',                                 'proverbs-en',         'Proverbs',                           'en'),
    ('https://librivox.org/isaiah-king-james-version/',                                   'isaiah-en',           'Isaiah',                             'en'),
    ('https://librivox.org/matthew-king-james-version/',                                  'matthew-en',          'Matthew',                            'en'),
    ('https://librivox.org/gospel-of-mark-king-james-version/',                           'mark-en',             'Mark',                               'en'),
    ('https://librivox.org/bible-kjv-nt-03-luke/',                                        'luke-en',             'Luke',                               'en'),
    ('https://librivox.org/john-by-king-james-version/',                                  'john-en',             'John',                               'en'),
    ('https://librivox.org/acts-king-james-version/',                                     'acts-en',             'Acts',                               'en'),
    ('https://librivox.org/romans-king-james-version/',                                   'romans-en',           'Romans',                             'en'),
    ('https://librivox.org/1-corinthians-king-james-version/',                            '1-corinthians-en',    '1 Corinthians',                      'en'),
    ('https://librivox.org/hebrews-king-james-version/',                                  'hebrews-en',          'Hebrews',                            'en'),
    ('https://librivox.org/book-of-james-kjv/',                                           'james-en',            'James',                              'en'),
    ('https://librivox.org/revelation-king-james-version/',                               'revelation-en',       'Revelation',                         'en'),
]

BATCH_MODEL = 'medium'
total = len(BATCH_BOOKS)
results = []

for i, (url, slug, book_dir, lang) in enumerate(BATCH_BOOKS):
    print(f'\n[{i+1}/{total}] {book_dir}')
    try:
        run_book(url, slug, book_dir, lang, BATCH_MODEL)
        results.append((book_dir, 'done'))
    except Exception as e:
        results.append((book_dir, f'ERROR: {e}'))
        print(f'  ERROR: {e} — skipping to next book')

print('\n' + '='*60)
print('BATCH COMPLETE')
print('='*60)
for name, status in results:
    icon = 'v' if status in ('done', 'Already done') else 'X'
    print(f'  [{icon}] {name}: {status}')

In [ ]:
# CELL 9 — Sanity-check merged VTTs already in Drive (run BEFORE downloading/committing)
# Catches broken transcriptions early: missing/dropped chapters, Whisper
# hallucination loops (repeated cues), and suspiciously short merges.
# Standalone: only needs Cells 4 + 5 run first (NOT Cell 8 — that starts the
# real batch). Safe to run in a separate Colab session from the active batch;
# it only reads from Drive, never writes.

# slug -> book_dir, copied from Cell 8's BATCH_BOOKS (kept separate so this
# cell never has to execute Cell 8's batch loop just to get the mapping)
_SLUG_TO_DIR = {
    'don-quijote-vol-1': 'Don Quijote Vol. I',
    'don-quijote-vol-2': 'Don Quijote Vol. II',
    'la-divina-comedia': 'La Divina Comedia',
    'crimen-y-castigo': 'Crimen y Castigo',
    'cuatro-jinetes': 'Los Cuatro Jinetes del Apocalipsis',
    'orgullo-y-prejuicio': 'Orgullo y Prejuicio',
    'pepita-jimenez': 'Pepita Jiménez',
    'platero-y-yo': 'Platero y yo',
    'viaje-al-centro': 'Viaje al Centro de la Tierra',
    'la-odisea': 'La Odisea',
    'cuentos-de-la-selva': 'Cuentos de la Selva',
    'fabulas-y-verdades': 'Fábulas y Verdades',
    'martin-fierro': 'El Gaucho Martín Fierro',
    'genesis-es': 'Génesis',
    'exodo-es': 'Éxodo',
    'mateo-es': 'Mateo',
    'marcos-es': 'Marcos',
    'lucas-es': 'Lucas',
    'juan-es': 'Juan',
    'romanos-es': 'Romanos',
    'hechos-es': 'Hechos',
    'apocalipsis-es': 'Apocalipsis',
    'proverbios-es': 'Proverbios',
    'isaias-es': 'Isaías',
    'salmos-es': 'Salmos',
    'santiago-es': 'Santiago',
    '1-corintios-es': '1 Corintios',
    'hebreos-es': 'Hebreos',
    'efesios-es': 'Efesios',
    'filipenses-es': 'Filipenses',
    'walden': 'Walden',
    'meditations': 'Meditations',
    'jane-eyre': 'Jane Eyre',
    'pride-and-prejudice': 'Pride and Prejudice',
    'treasure-island': 'Treasure Island',
    'dracula': 'Dracula',
    'anne-of-green-gables': 'Anne of Green Gables',
    'frankenstein': 'Frankenstein',
    'dorian-gray': 'The Picture of Dorian Gray',
    'scarlet-letter': 'The Scarlet Letter',
    'jekyll-hyde': 'The Strange Case of Dr Jekyll and Mr Hyde',
    'tom-sawyer': 'The Adventures of Tom Sawyer',
    'time-machine': 'The Time Machine',
    'psalms-en': 'Psalms',
    'proverbs-en': 'Proverbs',
    'isaiah-en': 'Isaiah',
    'matthew-en': 'Matthew',
    'mark-en': 'Mark',
    'luke-en': 'Luke',
    'john-en': 'John',
    'acts-en': 'Acts',
    'romans-en': 'Romans',
    '1-corinthians-en': '1 Corinthians',
    'hebrews-en': 'Hebrews',
    'james-en': 'James',
    'revelation-en': 'Revelation',
}

def sanity_check(merged_path):
    cues = parse_cues(merged_path.read_text(encoding='utf-8'))
    slug = merged_path.stem.replace('.merged', '')
    book_dir = _SLUG_TO_DIR.get(slug)
    chapters_expected = None
    if book_dir:
        src_dir = DRIVE_DIR / book_dir
        if src_dir.exists():
            chapters_expected = len(list(src_dir.glob('*.vtt')))

    if not cues:
        return {'slug': slug, 'cues': 0, 'mins': 0, 'chapters_found': 0,
                'chapters_expected': chapters_expected, 'flags': ['EMPTY MERGE']}

    duration_min = round(max(e for _, e, _ in cues) / 60, 1)
    # chapter boundaries = the >1.5s gaps the merge step inserts between files
    boundaries = sum(1 for (_, e0, _), (s1, _, _) in zip(cues, cues[1:]) if s1 - e0 > 1.5)
    chapters_found = boundaries + 1
    repeats = sum(1 for a, b in zip(cues, cues[1:]) if a[2].strip() == b[2].strip())

    flags = []
    if chapters_expected and chapters_found != chapters_expected:
        flags.append(f'chapter mismatch: merge has {chapters_found}, source has {chapters_expected}')
    if repeats:
        flags.append(f'{repeats} repeated/looping cue(s) — possible Whisper hallucination')
    if chapters_expected and chapters_expected > 2 and duration_min < 5:
        flags.append('suspiciously short for a multi-chapter book')

    return {'slug': slug, 'cues': len(cues), 'mins': duration_min,
            'chapters_found': chapters_found, 'chapters_expected': chapters_expected,
            'flags': flags}


merged_files = sorted(DRIVE_DIR.glob('*.merged.vtt'))
print(f'{len(merged_files)} merged VTT(s) in Drive\n')
print(f'{"slug":<24}{"cues":>6}{"mins":>8}{"chaps":>7}{"expect":>8}  flags')
print('-' * 80)
problems = []
for f in merged_files:
    r = sanity_check(f)
    expect = r['chapters_expected'] if r['chapters_expected'] is not None else '?'
    flag_str = '; '.join(r['flags']) if r['flags'] else 'OK'
    print(f'{r["slug"]:<24}{r["cues"]:>6}{r["mins"]:>8}{r["chapters_found"]:>7}{expect!s:>8}  {flag_str}')
    if r['flags']:
        problems.append(r['slug'])

print()
if problems:
    print(f'FLAGGED ({len(problems)}): {", ".join(problems)}')
    print('Delete the .merged.vtt for these in Drive and re-run them through Cell 8 before downloading/committing.')
else:
    print('No problems detected in the merges currently in Drive.')